# Finne viktige forklaringsvariabler vha. K-fold CV

## Imports/sessions

In [61]:
from snowflake.snowpark.functions import col, sum as sum_, max as max_, datediff, round as round_, year, month, when, lit, lag, avg, count, stddev, to_date, dayofweek, weekofyear, dayofmonth, quarter, last_day
from snowflake.snowpark import Window
import re
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# from dwh_tools.get_or_create_session import get_or_create_session
from snowflake.snowpark import Session
from snowflake.snowpark.context import get_active_session
import os
import requests
import gzip
import json
# from dwh_tools.get_or_create_session import get_or_create_session
pd.set_option('display.max_columns', None)
import time
import pywin

In [62]:
def get_or_create_session(schema: str = None) -> Session:
    """
    Returns the active Snowpark session if running inside Snowflake,
    otherwise creates one locally using Azure OAuth (interactive login if needed).
 
    Parameters
    ----------
    config_path : str
        Path to a JSON config file containing Snowflake connection parameters
        (account, warehouse, role, database, schema).
 
    Returns
    -------
    Session
        A Snowpark Session object.
    """
    # If running on snowflake
    if 'POSIT_PRODUCT' in os.environ:
        session = Session.builder.getOrCreate()
        session.sql("USE DATABASE PROD_FOR_SKADE_PRODUKT_ADHOC").collect()
        if schema:
            session.sql("USE SCHEMA " + schema).collect()
        else:
            session.sql("USE SCHEMA PRODUKT_WRITE_DEV").collect()
        session.sql("USE WAREHOUSE SKADE_VWH").collect()
 
        return session
 
    try:
        session = get_active_session()
        return session
    except Exception:
        import win32api
        if schema is None:
            schema = 'PRODUKT_WRITE_DEV'
 
        connection_parameters = {
            "server": "km28161.west-europe.azure.snowflakecomputing.com",
            "warehouse": "SKADE_VWH",
            "account": "VK82539-KLP",
            "database": "PROD_FOR_SKADE_PRODUKT_ADHOC",
            "schema" : schema,
            "user": win32api.GetUserNameEx(win32api.NameUserPrincipal),  
            "authenticator": "externalbrowser"
        }
       
        # Create the session
        session = Session.builder.configs(connection_parameters).create()
        return session

In [63]:
session = get_or_create_session()

## Data

### Lese data

In [64]:
# Kundesenter data
df_inngang = session.table('elh_write.inngangsdata').to_pandas()
df_inngang.columns = df_inngang.columns.str.lower()
# TIA data
df_info = session.table('inngangsdata_info').to_pandas()
df_info.columns = df_info.columns.str.lower()
# Værdata
df_weather = pd.read_csv("../../data/oslo_weather.csv", delimiter=";")
df_weather.columns = df_weather.columns.str.lower()

### Data-fiks, inngangsdata

In [65]:
df_inngang['ankomst_dato'] = pd.to_datetime(df_inngang['ankomst_dato'], format="%d.%m.%Y %H:%M:%S").dt.date
# Respons (behandlingstid) som sekunder
df_inngang['behandlingstid'] = (
    df_inngang['behandlingstid']
    .str.split(':')
    .apply(lambda x: int(x[0]) * 60 + int(x[1]))
)
# Også etterbehandlingstid som sekunder:
df_inngang['etterbehandlingstid'] = (
    df_inngang['etterbehandlingstid']
    .str.split(':')
    .apply(lambda x: int(x[0]) * 60 + int(x[1]))
)
df_inngang["behandlet"]=df_inngang["behandlet"].eq("Behandlet")
# Aggregering til dag-nivå
df_dag = (
    df_inngang.groupby("ankomst_dato", as_index=False)
    .agg(
        antall_samtaler=("unique_id", "size"),
        behandlet_andel=("behandlet", "mean"),
        tid_i_ko_snitt=("tid_i_ko", "mean"),
        behandlingstid_snitt=("behandlingstid", "mean"),
        total_behandlingstid=("behandlingstid", "sum"),
        etterbehandligstid_snitt=("etterbehandlingstid", "mean"), 
        total_etterbehandligstid=("etterbehandlingstid", "sum")
    ).
    sort_values("ankomst_dato")
    .reset_index(drop=True)
)

### Data-fiks, TIA-data

In [66]:
# Datetime format
df_info['hf_dato'] = pd.to_datetime(df_info['hf_dato'], format="%Y-%m-%d").dt.date

### Data-fiks, værdata

In [67]:
# Fiks tid-format
df_weather["tid(norsk normaltid)"]= pd.to_datetime(df_weather["tid(norsk normaltid)"], format="%d.%m.%Y").dt.date
# Fiks datatype
df_weather["nedbør (døgn)"] = df_weather["nedbør (døgn)"].str.replace(",", ".", regex=True).astype(float)
df_weather["middeltemperatur (døgn)"] = df_weather["middeltemperatur (døgn)"].str.replace(",", ".", regex=True).astype(float)
df_weather = df_weather.rename(columns={"nedbør (døgn)": "nedbør", "middeltemperatur (døgn)": "middeltemperatur"})

### Sammenslåing av dataframes

In [68]:
df_temp = pd.merge(df_dag, df_info, left_on="ankomst_dato", right_on="hf_dato", how="left")
df = pd.merge(df_temp, df_weather, left_on="ankomst_dato", right_on="tid(norsk normaltid)", how="left")


### Sammenslåing av dekninger

In [69]:
df["antall_nye_kunder_b30_ny"] = df["antall_nye_kunder_b30_mpb01_ny"] + df["antall_nye_kunder_b30_eph01_ny"] + df["antall_nye_kunder_b30_epf01_ny"] + df["antall_nye_kunder_b30_upr01_ny"]
df["antall_nye_kunder_b30_for"] = df["antall_nye_kunder_b30_mpb01_for"] + df["antall_nye_kunder_b30_eph01_for"] + df["antall_nye_kunder_b30_epf01_for"] + df["antall_nye_kunder_b30_upr01_for"]
df["antall_hf_b30_ny"] = df["antall_hf_b30_mpb01_ny"] + df["antall_hf_b30_eph01_ny"] + df["antall_hf_b30_epf01_ny"] + df["antall_hf_b30_upr01_ny"]
df["antall_hf_b30_for"] = df["antall_hf_b30_mpb01_for"] + df["antall_hf_b30_eph01_for"] + df["antall_hf_b30_epf01_for"]  + df["antall_hf_b30_upr01_for"]
df["stddev_premieendring_b30_for"] = (df["stddev_premieendring_b30_eph01_for"] + df["stddev_premieendring_b30_epf01_for"] + df["stddev_premieendring_b30_mpb01_for"] + df["stddev_premieendring_b30_upr01_for"]) / 4
df["snitt_premieendring_b30_for"] = (df["snitt_premieendring_b30_eph01_for"] + df["snitt_premieendring_b30_epf01_for"] + df["snitt_premieendring_b30_mpb01_for"] + df["snitt_premieendring_b30_upr01_for"]) / 4

### Tar totalen av ny og for også:

df["antall_nye_kunder_b30_tot"] = df["antall_nye_kunder_b30_ny"] + df["antall_nye_kunder_b30_for"]
df["antall_hf_b30_tot"] = df["antall_hf_b30_ny"] + df["antall_hf_b30_for"]

### Fjerning av unødvendige kolonner:

In [70]:
# Liste med kolonnenavn som kan fjernes fra df:
cols_to_remove = [
    "hf_dato",
    "antall_nye_kunder_b7_mpb01_ny",
    "antall_nye_kunder_f30_mpb01_ny",
    "antall_nye_kunder_f7_mpb01_ny",
    "antall_hf_b7_mpb01_ny",
    "antall_hf_f30_mpb01_ny",
    "antall_hf_f7_mpb01_ny",
    "antall_nye_kunder_b7_eph01_for",
    "antall_nye_kunder_f30_eph01_for",
    "antall_nye_kunder_f7_eph01_for",
    "antall_hf_b7_eph01_for",
    "stddev_premieendring_b7_eph01_for",
    "snitt_premieendring_b7_eph01_for",
    "antall_hf_f30_eph01_for",
    "stddev_premieendring_f30_eph01_for",
    "snitt_premieendring_f30_eph01_for",
    "antall_hf_f7_eph01_for",
    "stddev_premieendring_f7_eph01_for",
    "snitt_premieendring_f7_eph01_for",
    "antall_nye_kunder_b7_eph01_ny",
    "antall_nye_kunder_f30_eph01_ny",
    "antall_nye_kunder_f7_eph01_ny",
    "antall_hf_b7_eph01_ny",
    "antall_hf_f30_eph01_ny",
    "antall_hf_f7_eph01_ny",
    "antall_nye_kunder_b7_epf01_ny",
    "antall_nye_kunder_f30_epf01_ny",
    "antall_nye_kunder_f7_epf01_ny",
    "antall_hf_b7_epf01_ny",
    "antall_hf_f30_epf01_ny",
    "antall_hf_f7_epf01_ny",
    "antall_nye_kunder_b7_epf01_for",
    "antall_nye_kunder_f30_epf01_for",
    "antall_nye_kunder_f7_epf01_for",
    "antall_hf_b7_epf01_for",
    "stddev_premieendring_b7_epf01_for",
    "snitt_premieendring_b7_epf01_for",
    "antall_hf_f30_epf01_for",
    "stddev_premieendring_f30_epf01_for",
    "snitt_premieendring_f30_epf01_for",
    "antall_hf_f7_epf01_for",
    "stddev_premieendring_f7_epf01_for",
    "snitt_premieendring_f7_epf01_for",
    "antall_nye_kunder_b7_upr01_ny",
    "antall_nye_kunder_f30_upr01_ny",
    "antall_nye_kunder_f7_upr01_ny",
    "antall_hf_b7_upr01_ny",
    "antall_hf_f30_upr01_ny",
    "antall_hf_f7_upr01_ny",
    "antall_nye_kunder_b7_mpb01_for",
    "antall_nye_kunder_f30_mpb01_for",
    "antall_nye_kunder_f7_mpb01_for",
    "antall_hf_b7_mpb01_for",
    "stddev_premieendring_b7_mpb01_for",
    "snitt_premieendring_b7_mpb01_for",
    "antall_hf_f30_mpb01_for",
    "stddev_premieendring_f30_mpb01_for",
    "snitt_premieendring_f30_mpb01_for",
    "antall_hf_f7_mpb01_for",
    "stddev_premieendring_f7_mpb01_for",
    "snitt_premieendring_f7_mpb01_for",
    "antall_nye_kunder_b7_upr01_for",
    "antall_nye_kunder_f30_upr01_for",
    "antall_nye_kunder_f7_upr01_for",
    "antall_hf_b7_upr01_for",
    "stddev_premieendring_b7_upr01_for",
    "snitt_premieendring_b7_upr01_for",
    "antall_hf_f30_upr01_for",
    "stddev_premieendring_f30_upr01_for",
    "snitt_premieendring_f30_upr01_for",
    "antall_hf_f7_upr01_for",
    "stddev_premieendring_f7_upr01_for",
    "snitt_premieendring_f7_upr01_for",
    "navn", 
    "stasjon", 
    "tid(norsk normaltid)"
]
df = df.drop(columns=cols_to_remove)


In [75]:
df[["ankomst_dato", "er_helligdag", "er_dag_foer_helligdag", "er_dag_etter_helligdag", "ukedag"]].head(10)
print(df.columns)

Index(['ankomst_dato', 'antall_samtaler', 'behandlet_andel', 'tid_i_ko_snitt',
       'behandlingstid_snitt', 'total_behandlingstid',
       'etterbehandligstid_snitt', 'total_etterbehandligstid',
       'antall_nye_kunder_b30_mpb01_ny', 'antall_hf_b30_mpb01_ny',
       'antall_nye_kunder_b30_eph01_for', 'antall_hf_b30_eph01_for',
       'stddev_premieendring_b30_eph01_for',
       'snitt_premieendring_b30_eph01_for', 'antall_nye_kunder_b30_eph01_ny',
       'antall_hf_b30_eph01_ny', 'antall_nye_kunder_b30_epf01_ny',
       'antall_hf_b30_epf01_ny', 'antall_nye_kunder_b30_epf01_for',
       'antall_hf_b30_epf01_for', 'stddev_premieendring_b30_epf01_for',
       'snitt_premieendring_b30_epf01_for', 'antall_nye_kunder_b30_upr01_ny',
       'antall_hf_b30_upr01_ny', 'antall_nye_kunder_b30_mpb01_for',
       'antall_hf_b30_mpb01_for', 'stddev_premieendring_b30_mpb01_for',
       'snitt_premieendring_b30_mpb01_for', 'antall_nye_kunder_b30_upr01_for',
       'antall_hf_b30_upr01_for', 'stdde

## Velge variabler (CV)

In [78]:
import statsmodels.api as sm
import statsmodels.formula.api as smf


In [76]:
# Number of folds
K = 10
# Variabler inkludert fra start:
covs = ['antall_nye_kunder_b30_mpb01_ny', 'antall_hf_b30_mpb01_ny',
       'antall_nye_kunder_b30_eph01_for', 'antall_hf_b30_eph01_for',
       'stddev_premieendring_b30_eph01_for',
       'snitt_premieendring_b30_eph01_for', 'antall_nye_kunder_b30_eph01_ny',
       'antall_hf_b30_eph01_ny', 'antall_nye_kunder_b30_epf01_ny',
       'antall_hf_b30_epf01_ny', 'antall_nye_kunder_b30_epf01_for',
       'antall_hf_b30_epf01_for', 'stddev_premieendring_b30_epf01_for',
       'snitt_premieendring_b30_epf01_for', 'antall_nye_kunder_b30_upr01_ny',
       'antall_hf_b30_upr01_ny', 'antall_nye_kunder_b30_mpb01_for',
       'antall_hf_b30_mpb01_for', 'stddev_premieendring_b30_mpb01_for',
       'snitt_premieendring_b30_mpb01_for', 'antall_nye_kunder_b30_upr01_for',
       'antall_hf_b30_upr01_for', 'stddev_premieendring_b30_upr01_for',
       'snitt_premieendring_b30_upr01_for', 'aar', 'kvartal', 'maaned',
       'ukenummer', 'ukedag', 'dag_i_maaned', 'er_helg', 'helligdag',
       'er_helligdag', 'er_dag_foer_helligdag', 'er_dag_etter_helligdag',
       'middeltemperatur', 'nedbør', 'antall_nye_kunder_b30_ny',
       'antall_nye_kunder_b30_for', 'antall_hf_b30_ny', 'antall_hf_b30_for',
       'stddev_premieendring_b30_for', 'snitt_premieendring_b30_for',
       'antall_nye_kunder_b30_tot', 'antall_hf_b30_tot']

In [ ]:
from sklearn import linear_model
from sklearn import preprocessing
from sklearn import model_selection
from sklearn.metrics import mean_tweedie_deviance, make_scorer

In [ ]:
def fit_model(self, model_type, data, response, weight, model_variables, transforms, fold_id):
    self.model_type = model_type
    self.model_variables = model_variables
    data = data[self.model_variables]

    self.data = data[:1].copy()
    
    self.onehot_encode(data)
    data = self.apply_onehot_encoding(data)
    
    self.create_capping(data)
    data = self.apply_capping(data)

    self.transforms = transforms
    data = self.apply_transforms(data)
    
    self.normalize(data)
    data = self.apply_normalization(data)

    # Convert fold labels to the required list of lists
    folds = [(np.where(np.array(fold_id) != fold)[0],
                np.where(np.array(fold_id) == fold)[0]) for fold in np.unique(fold_id.tolist())]
    
    if(self.model_type == 'glm'):
        
                
        param_grid = {'alpha' : np.exp(np.linspace(-8, -2, 3))}
        
        self.model = linear_model.TweedieRegressor(link = 'log', power = 1.5, max_iter = 10000)
        
        self.cv = model_selection.GridSearchCV(self.model,
                                                param_grid = param_grid,
                                                cv = folds,
                                                scoring = make_scorer(mean_tweedie_deviance, power = 1, greater_is_better = False),
                                                n_jobs = -1)
        
        self.cv.fit(data, response, sample_weight = weight)

        self.model = self.cv.best_estimator_

        self.get_importance()